In [1]:
!pip install -q giotto-tda

!pip install gudhi

!apt-get install -y python3-geopandas
!pip install geopandas


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.6/554.6 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.8/455.8 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 33.0 MB/s eta 0:00:00
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  binfmt-support fonts-dejavu-core fonts-font-awesome fonts-lato fonts-lyx libclang-cpp11
  libffi-dev libimagequant0 liblbfgsb0 libllvm11 liblzo2-2 libpfm4 libraqm0 libspatialindex-c6
  libspatialindex-dev libspatialindex6 libxsimd-dev libz3-4 libz3-dev llvm-11 llvm-11-dev
  llvm-11-linker-tools llvm-11-runtime llvm-11-tools mailcap mime-support numba-doc
  python-babel-locale

In [9]:
import numpy as np
import geopandas as gpd
from gtda.homology import VietorisRipsPersistence
from scipy.stats import wasserstein_distance
from sklearn.preprocessing import StandardScaler
import pandas as pd

# Cargar el dataset principal
csv_path = "Valencia_2015_AlarmasMujer.csv"
df = pd.read_csv(csv_path)

# Obtener las columnas 'crime_lon' y 'crime_lat' como base para los datasets
lon_lat_data = df[['crime_lon', 'crime_lat']].values

# Parámetros del experimento
n = lon_lat_data.shape[0]  # Número de puntos en el dataset
m = 11   # Número de datasets a generar en cada experimento
j = 1000    # Número de repeticiones del experimento

# Función para agregar una columna de distancia aleatoria y dividirla por 1000
def add_random_distances_and_scale(data, min_dist, max_dist):
    distances = np.random.uniform(min_dist, max_dist, size=(data.shape[0], 1))
    distances /= 1000  # Dividir las distancias entre 1000
    return np.hstack([data, distances])

# Función para calcular el diagrama de persistencia
def calculate_persistence_diagram(data, VR):
    scaler = StandardScaler()  # Escalar los datos
    data_scaled = scaler.fit_transform(data)  # Ajustar y transformar los datos
    diagrams = VR.fit_transform([data_scaled])
    return diagrams[0][:, :2]  # Tomar solo las columnas de nacimiento y muerte

# Instanciar Vietoris-Rips una vez
VR = VietorisRipsPersistence()

# Función para calcular la distancia de Wasserstein entre dos diagramas
def wasserstein_distance_between_diagrams(diag1, diag2):
    births1, deaths1 = diag1[:, 0], diag1[:, 1]
    births2, deaths2 = diag2[:, 0], diag2[:, 1]
    return (wasserstein_distance(births1, births2) +
            wasserstein_distance(deaths1, deaths2))

# Función para calcular la distancia Wasserstein media entre varios diagramas
def calculate_mean_wasserstein_distance(diagrams):
    num_diagrams = len(diagrams)
    total_wasserstein_distance = 0
    num_comparisons = 0

    for i in range(num_diagrams):
        for j in range(i + 1, num_diagrams):
            total_wasserstein_distance += wasserstein_distance_between_diagrams(diagrams[i], diagrams[j])
            num_comparisons += 1

    return total_wasserstein_distance / num_comparisons if num_comparisons > 0 else 0

# Experimento para calcular la media final de las distancias Wasserstein a través de j repeticiones
mean_wasserstein_across_experiments = []

for experiment in range(j):
    diagrams = []
    for _ in range(m):
        # Usar las coordenadas de 'crime_lon' y 'crime_lat'
        random_points_with_distances = add_random_distances_and_scale(lon_lat_data, 0.100031, 4855.4071)

        # Calcular el diagrama de persistencia para estos puntos
        diagrams.append(calculate_persistence_diagram(random_points_with_distances, VR))

    # Calcular la media de la distancia Wasserstein en este experimento
    mean_wasserstein = calculate_mean_wasserstein_distance(diagrams)
    mean_wasserstein_across_experiments.append(mean_wasserstein)
    print(f"Experiment {experiment+1}/{j}: Mean Wasserstein Distance = {mean_wasserstein}")

# Calcular la media final de las distancias Wasserstein
final_mean_wasserstein = np.mean(mean_wasserstein_across_experiments)
print(f"\nFinal Mean Wasserstein Distance after {j} experiments: {final_mean_wasserstein}")


Experiment 1/1000: Mean Wasserstein Distance = 0.22301051547708867
Experiment 2/1000: Mean Wasserstein Distance = 0.1863670773044232
Experiment 3/1000: Mean Wasserstein Distance = 0.1824493720193071
Experiment 4/1000: Mean Wasserstein Distance = 0.20940639448340928
Experiment 5/1000: Mean Wasserstein Distance = 0.16829388062782902
Experiment 6/1000: Mean Wasserstein Distance = 0.2082691896387851
Experiment 7/1000: Mean Wasserstein Distance = 0.18503906330878792
Experiment 8/1000: Mean Wasserstein Distance = 0.18121042341086513
Experiment 9/1000: Mean Wasserstein Distance = 0.23151642307437886
Experiment 10/1000: Mean Wasserstein Distance = 0.22667389060801965
Experiment 11/1000: Mean Wasserstein Distance = 0.19110181435639573
Experiment 12/1000: Mean Wasserstein Distance = 0.21731385675403708
Experiment 13/1000: Mean Wasserstein Distance = 0.22217110520372757
Experiment 14/1000: Mean Wasserstein Distance = 0.23847962173145418
Experiment 15/1000: Mean Wasserstein Distance = 0.2543456710